# PAD000016 showcase

This notebook runs `olinkanalyze` on [PRIDE PAD000016](https://www.ebi.ac.uk/pride/archive/projects/PAD000016),
an Olink Explore HT NPX study of cyproterone acetate (CPA) or spironolactone (SPIRO)
gender-affirming hormone therapy and pregnancy.

The public files used here are two long-format NPX parquet plates plus the SDRF
sample annotation. They live in `data/PAD000016/` (gitignored). Install the
package first (`pip install -e .`), then run the cells in order.


## Setup


In [ ]:
%matplotlib inline

from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd

import olinkanalyze as oa

pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 140)

print("olinkanalyze", oa.__version__)


## Data files

PAD000016 is hosted on EBI FTP (the same files as the Globus PRIDE collection).
The cell below downloads only the files this notebook needs if they are not
already present. EBI FTP can take 30–40 seconds to start transferring.


In [ ]:
PRIDE_FTP = "https://ftp.pride.ebi.ac.uk/pride/data/archive/2025/09/PAD000016"
FILES = (
    "X201SC23110522-Z01-F004_NPX.parquet",
    "X201SC25020748-Z01-F001.parquet",
    "GAHT_pregnancy_metadata_SDRF.tsv",
)


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "olinkanalyze").is_dir():
            return candidate
    return here


DATA = repo_root() / "data" / "PAD000016"
DATA.mkdir(parents=True, exist_ok=True)

for name in FILES:
    dest = DATA / name
    if dest.exists() and dest.stat().st_size > 0:
        print("have", dest.name, dest.stat().st_size, "bytes")
        continue
    print("downloading", name, "...")
    urlretrieve(f"{PRIDE_FTP}/{name}", dest)
    print("saved", dest)

RUN1 = DATA / "X201SC23110522-Z01-F004_NPX.parquet"
RUN2 = DATA / "X201SC25020748-Z01-F001.parquet"
SDRF = DATA / "GAHT_pregnancy_metadata_SDRF.tsv"


## `read_npx`

Both plates are Explore HT NPX parquet. `read_npx` accepts parquet, runs
`check_npx`, and stores the log on `DataFrame.attrs`. Both files also contain
`PCNormalizedNPX`; `preferred_names` keeps `NPX` as the quantification column.


In [ ]:
PREFERRED = {"quant": "NPX"}

run1_raw = oa.read_npx(RUN1, preferred_names=PREFERRED)
run2_raw = oa.read_npx(RUN2, preferred_names=PREFERRED)

summary = pd.DataFrame(
    {
        "run1": [
            run1_raw["SampleID"].nunique(),
            run1_raw["OlinkID"].nunique(),
            run1_raw["Panel"].nunique(),
            run1_raw.shape[0],
        ],
        "run2": [
            run2_raw["SampleID"].nunique(),
            run2_raw["OlinkID"].nunique(),
            run2_raw["Panel"].nunique(),
            run2_raw.shape[0],
        ],
    },
    index=["samples", "assays", "panels", "rows"],
)
summary


In [ ]:
run1_raw.head()


## Sample annotation

The SDRF maps each study `SampleID` to phenotype and plate (`run 1` / `run 2`).
Control wells (`SC*`, `IPC*`, `NC*`) are not in the SDRF and stay unlabeled.
Assigning columns in place keeps the `check_npx` log attached.


In [ ]:
sdrf = pd.read_csv(SDRF, sep="\t").rename(
    columns={
        "source name": "SampleID",
        "factor value[phenotype]": "Phenotype",
        "assay name": "Run",
    }
)[["SampleID", "Phenotype", "Run"]]

print(sdrf["Phenotype"].value_counts().to_string())
print()
print(sdrf["Run"].value_counts().to_string())


def add_sample_annotation(df: pd.DataFrame, annotation: pd.DataFrame) -> pd.DataFrame:
    keyed = annotation.drop_duplicates("SampleID").set_index("SampleID")
    sample_id = df["SampleID"].astype(str)
    df["Phenotype"] = sample_id.map(keyed["Phenotype"])
    df["Run"] = sample_id.map(keyed["Run"])
    return df


run1_raw = add_sample_annotation(run1_raw, sdrf)
run2_raw = add_sample_annotation(run2_raw, sdrf)


## QC (`check_npx`, `extract_check_log`)

`check_npx` reports unrecognized OlinkIDs, all-NA assays, duplicate sample IDs,
and assays with QC warnings. `extract_check_log` reads the log already stored
on the frame.


In [ ]:
log1 = oa.extract_check_log(run1_raw)
log2 = oa.extract_check_log(run2_raw)

qc_overview = pd.DataFrame(
    {
        "run1": [
            log1["col_names"]["quant"],
            log1["col_names"]["qc_warning"],
            len(log1["oid_invalid"]),
            len(log1["assay_na"]),
            len(log1["sample_id_dups"]),
            len(log1["assay_qc"]),
        ],
        "run2": [
            log2["col_names"]["quant"],
            log2["col_names"]["qc_warning"],
            len(log2["oid_invalid"]),
            len(log2["assay_na"]),
            len(log2["sample_id_dups"]),
            len(log2["assay_qc"]),
        ],
    },
    index=["quant column", "sample QC column", "invalid OlinkID", "all-NA assays", "duplicate SampleID", "assay QC warnings"],
)
print("assays with AssayQC warnings (run 1):", log1["assay_qc"])
print("assays with AssayQC warnings (run 2):", log2["assay_qc"])
qc_overview


## `clean_npx`

Drops control assays and control samples, assay QC warnings, and sample QC
failures, then re-runs the check log.


In [ ]:
run1 = oa.clean_npx(run1_raw)
run2 = oa.clean_npx(run2_raw)

print("run1", run1_raw.shape, "->", run1.shape, "samples", run1["SampleID"].nunique(), "assays", run1["OlinkID"].nunique())
print("run2", run2_raw.shape, "->", run2.shape, "samples", run2["SampleID"].nunique(), "assays", run2["OlinkID"].nunique())
run1["SampleType"].value_counts()


## Distribution and QC plots

These parquet files use `SampleQC` rather than `QC_Warning`. Pass that name
to `color_g`. The NPX distribution plot is shown on a few samples per
phenotype so the boxes stay readable; the QC scatter uses the full first plate.


In [ ]:
oa.set_plot_theme()

dist_ids = (
    run1.dropna(subset=["Phenotype"])
    .drop_duplicates("SampleID")
    .groupby("Phenotype", observed=True)["SampleID"]
    .head(3)
)
oa.olink_dist_plot(run1.loc[run1["SampleID"].isin(dist_ids)], color_g="Phenotype")


In [ ]:
oa.olink_qc_plot(run1_raw, color_g="SampleQC")


## PCA and heatmap

`olink_pca_plot` pivots samples × assays and colors points by a sample-level
column. `olink_heatmap_plot` shows the most variable assays, median-centered.


In [ ]:
oa.olink_pca_plot(run1, color_g="Phenotype")


In [ ]:
oa.olink_heatmap_plot(run1, n_assays=20)


## Intensity normalization

The two plates share only control well names (`IPC*`, `SC*`, `NC*`), not
study samples. After `clean_npx` those controls are gone, so intensity
(median) normalization is the matching method: it shifts run 2 so per-assay
medians match run 1.


In [ ]:
npx = oa.olink_normalization(
    df1=run1,
    df2=run2,
    df1_project_nr="run1",
    df2_project_nr="run2",
    reference_project="run1",
)

print(npx.groupby("Project", observed=True)["SampleID"].nunique())
print()
print(
    npx.dropna(subset=["Phenotype"])
    .drop_duplicates("SampleID")["Phenotype"]
    .value_counts()
    .to_string()
)
npx.head()


## `olink_ttest` and volcano

Welch t-tests of CPA GAHT baseline versus 6 months, excluding the run-2
technical replicates (`*_rep`). `olink_volcano_plot` expects the t-test table.


In [ ]:
cpa = npx.loc[
    npx["Phenotype"].isin(["CPA_GAHT_baseline", "CPA_GAHT_6_months"])
    & ~npx["SampleID"].astype(str).str.endswith("_rep")
].copy()

print(cpa.drop_duplicates("SampleID")["Phenotype"].value_counts().to_string())

ttest = oa.olink_ttest(cpa, variable="Phenotype")
print("significant (FDR < 0.05):", int((ttest["Threshold"] == "Significant").sum()), "of", len(ttest))
ttest.head(10)


In [ ]:
oa.olink_volcano_plot(ttest, olinkid_list=ttest["OlinkID"].head(12))


## Wilcoxon and paired t-test

`olink_wilcox` is the rank-sum analogue of the two-group t-test. Sample IDs
encode subject and time (`AA17_0m` / `AA17_6m`), so a paired t-test uses
`pair_id="Subject"`.


In [ ]:
wilcox = oa.olink_wilcox(cpa, variable="Phenotype")
print("Wilcoxon significant:", int((wilcox["Threshold"] == "Significant").sum()), "of", len(wilcox))

cpa["Subject"] = cpa["SampleID"].astype(str).str.rsplit("_", n=1).str[0]
paired = oa.olink_ttest(cpa, variable="Phenotype", pair_id="Subject")
print("paired t-test significant:", int((paired["Threshold"] == "Significant").sum()), "of", len(paired))

compare = (
    ttest.set_index("OlinkID")[["Assay", "estimate", "Adjusted_pval"]]
    .join(wilcox.set_index("OlinkID")["Adjusted_pval"].rename("wilcox_FDR"))
    .join(paired.set_index("OlinkID")["Adjusted_pval"].rename("paired_FDR"))
    .sort_values("Adjusted_pval")
)
compare.head(8)


## ANOVA

One-way ANOVA across the five largest phenotype groups, then pairwise Welch
post-hoc tests.


In [ ]:
anova_groups = [
    "CPA_GAHT_baseline",
    "CPA_GAHT_6_months",
    "SPIRO_GAHT_baseline",
    "SPIRO_GAHT_6_months",
    "pregnancy_trimester_3",
]
anova_df = npx.loc[
    npx["Phenotype"].isin(anova_groups)
    & ~npx["SampleID"].astype(str).str.endswith("_rep")
].copy()

print(anova_df.drop_duplicates("SampleID")["Phenotype"].value_counts().to_string())

anova = oa.olink_anova(anova_df, variable="Phenotype")
print("ANOVA significant:", int((anova["Threshold"] == "Significant").sum()), "of", len(anova))
anova.head()


In [ ]:
posthoc = oa.olink_anova_posthoc(anova_df, variable="Phenotype")
print(posthoc["contrast"].value_counts().to_string())
posthoc.sort_values("Adjusted_pval").head(8)


## LOD

`olink_lod` flags NPX below the assay LOD. These Explore HT parquet files do
not include an LOD column, so the call is expected to fail. Use a Standard
NPX export (or the optional `Olink_CPA_SPIRO_GAHT_and_pregnancy.npx.csv` from
the same PRIDE dataset) if you need BelowLOD flags.


In [ ]:
log = oa.extract_check_log(run1)
print("LOD in check_log:", "lod" in log["col_names"])
try:
    oa.olink_lod(run1)
except ValueError as exc:
    print(exc)
